In [1]:
"""
Climate-Induced Migration Pressure Modeling — India (District-Level)
Step 3: Schema Alignment & Sequential Merge
"""
 
import pandas as pd
import os

import sys
sys.path.append("C:/Users/asus/OneDrive/Documents/climate-migration-prediction/src")
from step02_preprocessing import (       # noqa: E402  (import after sys.path fix)
    preprocess_census,
    preprocess_rainfall,
    preprocess_agriculture,
    preprocess_mpi,
    CENSUS_PATH, RAINFALL_PATH, MPI_PATH, AGRI_PATH,
)
 
JOIN_KEYS = ["state_key", "district_key"]
DIVIDER   = "=" * 70

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Merge helpers
# ─────────────────────────────────────────────────────────────────────────────
 
def _new_columns(base_cols: list[str], merged_cols: list[str]) -> list[str]:
    """Return columns that appear in merged but not in base (excluding suffixes)."""
    base_set = set(base_cols)
    return [c for c in merged_cols if c not in base_set and not c.endswith(("_x", "_y"))]
 
 
def left_join_with_audit(
    left: pd.DataFrame,
    right: pd.DataFrame,
    label: str,
) -> pd.DataFrame:
    """
    Left-join `right` onto `left` on JOIN_KEYS.
    Prints match/unmatch counts and missing value summary for newly added columns.
    Drops duplicate key columns from `right` that already exist in `left`.
    """
    print(f"\n{'─' * 60}")
    print(f"  Merging: Census base ← {label}")
    print(f"{'─' * 60}")
 
    base_cols = left.columns.tolist()
 
    # Drop columns from right that already exist in left (except join keys)
    right_only = [c for c in right.columns if c not in left.columns or c in JOIN_KEYS]
    right_trimmed = right[right_only]
 
    merged = left.merge(right_trimmed, on=JOIN_KEYS, how="left", indicator=True)
 
    matched   = (merged["_merge"] == "both").sum()
    unmatched = (merged["_merge"] == "left_only").sum()
 
    print(f"  Matched rows   : {matched:>4}  ({matched / len(left) * 100:.1f}% of base)")
    print(f"  Unmatched rows : {unmatched:>4}  ({unmatched / len(left) * 100:.1f}% of base — NaN filled)")
 
    merged = merged.drop(columns=["_merge"])
 
    new_cols = _new_columns(base_cols, merged.columns.tolist())
    missing  = merged[new_cols].isnull().sum()
    missing  = missing[missing > 0]
 
    print(f"\n  Missing values in newly joined columns:")
    if missing.empty:
        print("    ✓ None")
    else:
        for col, n in missing.items():
            print(f"    {col:<35} {n:>4}  ({n / len(merged) * 100:.1f}%)")
 
    return merged.reset_index(drop=True)
 

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Main merge pipeline
# ─────────────────────────────────────────────────────────────────────────────
 
def merge_all(
    df_census:   pd.DataFrame,
    df_rainfall: pd.DataFrame,
    df_agri:     pd.DataFrame,
    df_mpi:      pd.DataFrame,
) -> pd.DataFrame:
    """
    Sequentially left-join Rainfall, Agriculture, and MPI onto Census.
    Census is the spine: its 640 district rows are never dropped.
    """
    print(f"\n{DIVIDER}")
    print("  MERGE PIPELINE")
    print(DIVIDER)
 
    df = df_census.copy()
 
    # ── Merge 1: Census ← Rainfall ────────────────────────────────────────
    df = left_join_with_audit(df, df_rainfall, "Rainfall")
 
    # ── Merge 2: Census+Rainfall ← Agriculture ────────────────────────────
    df = left_join_with_audit(df, df_agri, "Agriculture")
 
    # ── Merge 3: Census+Rainfall+Agriculture ← MPI ────────────────────────
    df = left_join_with_audit(df, df_mpi, "MPI")
 
    return df
 
 
def final_audit(df: pd.DataFrame) -> None:
    """Print shape, duplicate check, and full missing value summary."""
    print(f"\n{DIVIDER}")
    print("  FINAL MERGED DATASET — AUDIT")
    print(DIVIDER)
 
    print(f"\n[Shape]")
    print(f"  {df.shape[0]} rows  ×  {df.shape[1]} columns")
 
    print(f"\n[Duplicate districts]")
    dupes = df.duplicated(subset=JOIN_KEYS)
    if dupes.any():
        print(f"  ⚠  {dupes.sum()} duplicate (state_key, district_key) pairs found:")
        print(df[dupes][["state", "district"]].to_string(index=False))
    else:
        print("  ✓  No duplicate districts.")
 
    print(f"\n[Missing value summary — all columns]")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    summary = pd.DataFrame({"missing_n": missing, "missing_pct": missing_pct})
    summary = summary[summary["missing_n"] > 0].sort_values("missing_n", ascending=False)
 
    if summary.empty:
        print("  ✓  No missing values in the merged dataset.")
    else:
        print(f"  {'Column':<42} {'N missing':>10}  {'%':>6}")
        print(f"  {'─' * 42} {'─' * 10}  {'─' * 6}")
        for col, row in summary.iterrows():
            print(f"  {col:<42} {int(row['missing_n']):>10}  {row['missing_pct']:>5.1f}%")
 
    print(f"\n[Column listing]")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:>2}. {col}")
 
 

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────────────────────────────────────
 
def main() -> pd.DataFrame:
    print("\nRunning preprocessing …")
    df_census   = preprocess_census(CENSUS_PATH)
    df_rainfall = preprocess_rainfall(RAINFALL_PATH)
    df_agri     = preprocess_agriculture(AGRI_PATH)
    df_mpi      = preprocess_mpi(MPI_PATH)
 
    df_merged = merge_all(df_census, df_rainfall, df_agri, df_mpi)
 
    final_audit(df_merged)
 
    print(f"\n{DIVIDER}")
    print("  Merge complete. Missing values preserved — no rows dropped.")
    print(DIVIDER)
 
    return df_merged
 
 
if __name__ == "__main__":
    df_merged = main()


Running preprocessing …


C:\Users/asus/OneDrive/Documents/climate-migration-prediction/src\step02_preprocessing.py:212: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df.groupby(["state", "district"]).apply(lambda g: pd.Series({



[MPI Columns]:
['state', 'district', 'country', 'mpi_data_source', 'population_share_of_the_district', 'multidimensional_poverty_index_mpi_of_the_country', 'multidimensional_poverty_of_the_districts_multidimensional_poverty_index_mpi__ha', 'multidimensional_poverty_of_the_districts_headcount_ratio_population_in_multidimensional_poverty_h', 'multidimensional_poverty_of_the_districts_intensity_of_deprivation_among_the_poor_a', 'multidimensional_poverty_of_the_districts_total_population_2016ᵃ', 'multidimensional_poverty_of_the_districts', 'number_of_mpi_poor_people']
[MPI] rows: 640 | cols: 7
Series([], dtype: int64)

  MERGE PIPELINE

────────────────────────────────────────────────────────────
  Merging: Census base ← Rainfall
────────────────────────────────────────────────────────────
  Matched rows   :  475  (74.2% of base)
  Unmatched rows :  165  (25.8% of base — NaN filled)

  Missing values in newly joined columns:
    rainfall_mean                        165  (25.8%)
    rainfa

In [5]:
print("Census duplicates:", df_census.duplicated(["state_key","district_key"]).sum())
print("Rainfall duplicates:", df_rainfall.duplicated(["state_key","district_key"]).sum())
print("Agriculture duplicates:", df_agri.duplicated(["state_key","district_key"]).sum())
print("MPI duplicates:", df_mpi.duplicated(["state_key","district_key"]).sum())

NameError: name 'df_census' is not defined

In [ ]:
print("CENSUS:")
print(df_census[["state", "district"]].head(10))

print("\nRAINFALL:")
print(df_rainfall[["state", "district"]].head(10))